# Portfolio Analytics & Equity Evaluation Framework
**Role:** Data Business Analyst  
**Objective:** Systematic tracking, quantitative evaluation, and data-driven decision-making for equity investments.

## 1. Executive Summary & Investment Thesis
*   **Company Name / Ticker:** [e.g., Apple Inc. / AAPL]
*   **Analyst Name:** [Your Name]
*   **Date:** June 2026
*   **Core Thesis:** Brief summary of why this stock is being analyzed (e.g., undervalued tech leader, strong dividend growth, structural market shift).

## 2. Data Acquisition & Pipeline Setup
*Goal: Import necessary libraries and establish robust data connections.*

### 2.1 Dependencies & Environment
```python
# Libraries to be loaded: yfinance, pandas, numpy, matplotlib/plotly, scipy

In [1]:
# SECTION 2.1: Dependencies & Environment Setup (with Auto-Installation)

import sys
import subprocess

# List of required packages for our Business Analytics Pipeline
required_packages = {
    'yfinance': 'yfinance',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'scipy': 'scipy',
    'statsmodels': 'statsmodels',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'plotly': 'plotly',
    'socket': 'socket',
    'lxml': 'lxml',
    'pytickersymbols': 'pytickersymbols',
}

print(">> Checking and preparing pipeline dependencies...")

for module_name, pip_name in required_packages.items():
    try:
        __import__(module_name)
    except ModuleNotFoundError:
        print(f"   [!] '{module_name}' not found. Installing {pip_name} via pip...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
            print(f"   [✓] Successfully installed {pip_name}.")
        except Exception as e:
            print(f"   [X] Failed to install {pip_name}. Error: {e}")

# Re-verify and final import of all dependencies
import yfinance as yf
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import socket
import pickle
import os
import time
import random
import urllib.request
from datetime import datetime
from pytickersymbols import PyTickerSymbols
import sqlite3
import re

# System & Settings
warnings.filterwarnings('ignore')

# Inline plotting configuration for Jupyter Notebooks
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic('matplotlib', 'inline')
except Exception:
    pass

# Setting a clean aesthetic style for matplotlib/seaborn
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

print("\n>> All pipeline dependencies successfully verified and loaded.")

>> Checking and preparing pipeline dependencies...

>> All pipeline dependencies successfully verified and loaded.


### 2.2 Core Market Data Ingestion
Historical Price Data (Daily/Weekly/Monthly Close, Adjusted Close, Volume)

Fundamental Data (Balance Sheet, Income Statement, Cash Flow)

In [2]:
# Download Pipeline Konfiguration
market_switches = {
    "AKTIEN": 0,            # US Bluechips (S&P 500 Marktführer)
    "ETF": 0,                # Globale Core-ETFs (World, Emerging Markets, S&P500)
    "ETF_SEKTOREN": 0,      # Branchen-ETFs (Tech, Healthcare, Energy, Financials)
    "ROHSTOFFE": 0,         # Edelmetalle, Energieträger und Agrar-Futures
    "KRYPTO": 0,            # Top Large-Cap Kryptowährungen
    "FOREX": 0,             # Globale Hauptwährungspaare (Majors)
    "INDIZES": 0,           # Weltleitindizes (DAX, S&P 500, Nasdaq, Nikkei)
    "ANLEIHEN": 0,          # Staatsanleihen / Bonds (US Treasury Yields)
}

def find_project_path(anchor_name="offline_ai", target_name="temp"):
    """
    Wandert fehlersicher (case-insensitive) nach oben, um den echten Projekt-Hauptordner zu finden.
    Sucht danach das gesamte Projektsystem nach einem existierenden Zielordner ab,
    bevor ein neuer erstellt wird.
    """
    # Startpunkt ist das aktuelle Verzeichnis des Notebooks
    current_path = os.path.abspath(os.getcwd())
    project_root = None
    
    # STUFE 1: Rigorose Aufwärtssuche (Unabhängig von Groß-/Kleinschreibung)
    while True:
        # Vergleich in Kleinbuchstaben, damit 'Offline_AI' und 'offline_ai' matchen
        if os.path.basename(current_path).lower() == anchor_name.lower():
            project_root = current_path
            break
        
        parent = os.path.dirname(current_path)
        if parent == current_path:  # System-Wurzel erreicht
            break
        current_path = parent
        
    # Fallback-Sicherung, falls der Anker mitten im Pfad liegt
    if not project_root:
        parts = os.path.abspath(os.getcwd()).split(os.sep)
        for i in range(len(parts), 0, -1):
            if parts[i-1].lower() == anchor_name.lower():
                project_root = os.sep.join(parts[:i])
                break

    # Ultimativer Notanker, falls gar nichts matcht
    if not project_root:
        project_root = os.path.abspath(os.getcwd())
        print(f"[!] Projekt-Anker '{anchor_name}' physisch nicht gefunden. Nutze Basis: {project_root}")
    else:
        print(f"[✓] Projekthauptordner lokalisiert unter: {project_root}")

    # STUFE 2: Globale Abwärtssuche im gesamten Projektbaum nach EXISTIERENDEM Zielordner
    # Wir durchsuchen JEDEN Winkel, damit kein doppelter Ordner entsteht
    for root, dirs, files in os.walk(project_root):
        # Versteckte Verzeichnisse (z.B. .git oder .ipynb_checkpoints) überspringen
        if any(part.startswith('.') for part in root.split(os.sep)):
            continue
            
        for d in dirs:
            if d.lower() == target_name.lower():
                found_path = os.path.join(root, d)
                print(f"[✓] Existierenden Ordner '{target_name}' im System gefunden: {found_path}")
                return found_path
                
    # STUFE 3: Erst wenn die Stufe 2 komplett fehlschlägt, greift die Automatik am Hauptstamm
    target_path = os.path.join(project_root, target_name)
    try:
        os.makedirs(target_path, exist_ok=True)
        print(f"[✓] Ordner '{target_name}' existierte nirgends. Neu angelegt im Hauptstamm: {target_path}")
        return target_path
    except Exception as e:
        print(f"[!] Fehler beim Erstellen des Ordners '{target_name}': {e}")
        return project_root

def _is_cache_from_today(cache_file_path):
    if not os.path.exists(cache_file_path):
        return False
    file_date = datetime.fromtimestamp(os.path.getmtime(cache_file_path)).date()
    return file_date == datetime.now().date()

def _has_internet():
    try:
        socket.setdefaulttimeout(3)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(("8.8.8.8", 53))
        return True
    except socket.error:
        return False

def apply_rate_limit():
    delay = random.uniform(1.5, 4.5)
    print(f"   [i] Rate Limiting: Pausiere für {delay:.2f} Sekunden...")
    time.sleep(delay)

def update_daily_delta(ticker, cached_payload):
    if not cached_payload or 'prices' not in cached_payload:
        return "5y"
    prices_df = cached_payload['prices']
    if prices_df.empty:
        return "5y"
    last_recorded_date = prices_df.index.max().date()
    today = datetime.now().date()
    if last_recorded_date >= today:
        return None
    start_fetch_date = last_recorded_date + datetime.timedelta(days=1)
    return start_fetch_date.strftime('%Y-%m-%d')

def save_live_directory(cache_file_path, data_universe):
    """
    Schreibt das gesamte Daten-Universum live auf die Festplatte.
    Überprüft nach dem Schreiben direkt, ob die Datei existiert und Inhalt hat.
    """
    try:
        # 1. Schreibvorgang auf Disk
        with open(cache_file_path, 'wb') as f:
            pickle.dump(data_universe, f)
            
        # 2. Physische Überprüfung auf der Festplatte (Hardware-Rückmeldung)
        if os.path.exists(cache_file_path):
            file_size = os.path.getsize(cache_file_path)
            
            if file_size > 0:
                print(f"[✓] Bestätigt: Datei existiert und ist beschrieben ({file_size} Bytes): {cache_file_path}")
            else:
                print(f"[!] Warnung: Datei wurde erstellt, ist aber LEER (0 Bytes): {cache_file_path}")
        else:
            print(f"[!] Kritisch: Schreibvorgang ohne Fehler beendet, aber Datei existiert nicht auf Disk!")
            
    except Exception as e:
        print(f"[!] Kritischer Fehler bei der Live-Datenablage: {e}")

def fetch_bulk_price_data(ticker_list, period="5y", start_date=None, end_date=None):
    """
    KLEMME 4.6: Sendet ein einziges massives Paket an die API.
    Lädt die historischen Kurse für ALLE Ticker gleichzeitig und formatiert 
    sie vollständig einsatzbereit (inkl. Datetime-Index und Adj Close).
    
    Rückgabetyp: Dictionary {ticker_string: DataFrame_oder_None}
    """
    if not ticker_list:
        return {}
    
    ticker_string = " ".join(ticker_list)
    print(f"   [API Bulk] Starte Paket-Download für {len(ticker_list)} Ticker...")
    
    formatted_bulk_data = {}
    
    try:
        if start_date and end_date:
            bulk_df = yf.download(ticker_string, start=start_date, end=end_date, group_by="ticker", progress=False)
        else:
            bulk_df = yf.download(ticker_string, period=period, group_by="ticker", progress=False)
            
        # --- FALL A: Sonderfall-Absicherung (Nur ein einziger Ticker abgefragt) ---
        if len(ticker_list) == 1:
            ticker = ticker_list[0]
            if not bulk_df.empty:
                raw_df = bulk_df.copy()
                raw_df.index = pd.to_datetime(raw_df.index)
                prices = raw_df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
                prices['Adj Close'] = raw_df['Adj Close'] if 'Adj Close' in raw_df.columns else raw_df['Close']
                formatted_bulk_data[ticker] = prices
            else:
                formatted_bulk_data[ticker] = None
            return formatted_bulk_data

        # --- FALL B: Standard-Verarbeitung (Multi-Index DataFrame bei mehreren Tickern) ---
        for ticker in ticker_list:
            if ticker in bulk_df.columns.levels[0]:
                raw_ticker_df = bulk_df[ticker].dropna(how='all').copy()
                
                if not raw_ticker_df.empty:
                    # 1. Datetime-Index erzwingen
                    raw_ticker_df.index = pd.to_datetime(raw_ticker_df.index)
                    
                    # 2. Struktur exakt nach Standard aufbauen
                    prices = raw_ticker_df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
                    
                    # Logik-Korrektur: Sicheres Setzen des Adj Close Spaltennamens
                    prices['Adj Close'] = raw_ticker_df['Adj Close'] if 'Adj Close' in raw_ticker_df.columns else raw_ticker_df['Close']
                        
                    formatted_bulk_data[ticker] = prices
                else:
                    formatted_bulk_data[ticker] = None
            else:
                formatted_bulk_data[ticker] = None
                
        return formatted_bulk_data
        
    except Exception as e:
        print(f"   [!] Kritischer Fehler beim API-Bulk-Abruf oder Formatierung: {e}")
        return None


def fetch_single_fundamentals(ticker, category):
    """
    KLEMME 4.7: Holt die tiefen Fundamentaldaten (Bilanzen, Cashflow) für ein einzelnes Asset.
    Vollständig in sich geschlossen und liefert die exakte Payload-Struktur.
    """
    import yfinance as yf
    import pandas as pd
    
    try:
        t_obj = yf.Ticker(ticker)
        payload = {
            'balance_sheet': pd.DataFrame(),
            'income_statement': pd.DataFrame(),
            'cash_flow': pd.DataFrame(),
            'info': {},
            'category': category,  # Direkt hier verankert für vollständige Funktionalität
            'has_fundamentals': False
        }
        
        # Info-Struktur absichern
        try:
            payload['info'] = t_obj.info
        except Exception:
            payload['info'] = {'error': 'No info retrieved'}
            
        # Fundamentaldaten nur bei Aktien abrufen
        if category == "AKTIEN":
            try:
                payload['balance_sheet'] = t_obj.balance_sheet
                payload['income_statement'] = t_obj.financials
                payload['cash_flow'] = t_obj.cashflow
                payload['has_fundamentals'] = True
            except Exception as e:
                print(f"      [!] Fehler beim Abruf der Tabellen (Bilanzen/Financials) für {ticker}: {e}")
            
        return payload
    except Exception as e:
        print(f"      [!] Fehler beim Laden der Fundamentaldaten für {ticker}: {e}")
        return None

def _discover_active_markets(market_switches):
    """
    KLEMME 4: Ermittelt die Ticker-Begriffe rein dynamisch aus Bibliotheken.
    Verarbeitet die Mengen-Begrenzung (0 = unbegrenzt).
    
    MODIFIKATION: Integrierter Wikipedia-Live-Scrape für S&P 500 Ticker.
    Wird nur einmalig pro Durchlauf aufgerufen, um Überlastung zu vermeiden.
    Kappt Suffixe dynamisch auf den reinen US-Stamm-Ticker.
    
    FIX: Schaltet einen validen User-Agent vor, um HTTP Error 403 (Forbidden) abzufangen.
    """
    import urllib.request
    
    active_tickers = {}
    ticker_db = None
    
    try:
        from pytickersymbols import PyTickerSymbols
        ticker_db = PyTickerSymbols()
    except Exception:
        print("[!] PyTickerSymbols-Bibliothek nicht verfügbar. Fallback aktiv.")

    for kategorie, schalterwert in market_switches.items():
        ticker_liste = []
        
        if kategorie == "AKTIEN":
            # 1. OPTION: Primärer Versuch über den ressourcenschonenden Wikipedia-Scrape (Einmalige Abfrage)
            try:
                url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
                
                # Header-Injektion gegen den 403-Block der Wikipedia-Server
                req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req) as response:
                    html_content = response.read()
                
                tables = pd.read_html(html_content)
                sp500_table = tables[0]
                # Punkte direkt durch Bindestriche ersetzen (z.B. BRK.B -> BRK-B)
                ticker_liste = sp500_table['Symbol'].str.upper().str.replace('.', '-', regex=False).tolist()
            except Exception as e:
                print(f"[!] Wikipedia-Scrape übersprungen oder fehlgeschlagen ({e}). Weiche auf lokale Library aus.")
                ticker_liste = []

            # 2. OPTION: Lokaler Fallback über PyTickerSymbols, falls Wikipedia fehlschlägt
            if not ticker_liste and ticker_db is not None:
                try:
                    sp500_stocks = ticker_db.get_stocks_by_index('S&P 500')
                    
                    for stock in sp500_stocks:
                        if stock.get('symbols') and len(stock['symbols']) > 0 and 'yahoo' in stock['symbols'][0]:
                            raw_yahoo = stock['symbols'][0]['yahoo'].upper()
                            
                            # Matcht nur die ersten Alphanumerischen Zeichen vor einem Punkt oder Bindestrich
                            match = re.match(r'^([A-Z0-9]+)', raw_yahoo)
                            if match:
                                ticker_liste.append(match.group(1))
                            else:
                                ticker_liste.append(raw_yahoo.replace('.', '-'))
                                
                except Exception as e:
                    print(f"[!] Fehler beim Abruf von PyTickerSymbols für {kategorie}: {e}")
                    continue
        else:
            # Platzhalter für deine anderen System-Bibliotheken (Krypto, ETF etc.)
            continue

        if not ticker_liste:
            continue

        # Mengen-Begrenzung anwenden
        if schalterwert == 0:
            auswahl = ticker_liste
            print(f"   [Scout] {kategorie}: Schalter steht auf 0 (Unbegrenzt). {len(auswahl)} Begriffe bereitgestellt.")
        elif schalterwert > 0:
            auswahl = ticker_liste[:schalterwert]
            print(f"   [Scout] {kategorie}: Begrenzung aktiv. Übergebe {len(auswahl)} von {len(ticker_liste)} Begriffen.")
        else:
            auswahl = []

        for ticker in auswahl:
            active_tickers[ticker] = kategorie

    return active_tickers

# Central Pipeline Module

def run_market_pipeline_download(market_switches):
    """
    Der zentrale Klemmblock. Koordiniert den Datenfluss über hocheffiziente
    Paket-Downloads (Bulk) und sichert fehlerhafte Ticker persistent ab.
    """
    
    print("[START] Starte globale Markt-Pipeline (Bulk-Architektur)")
    
    # KLEMME 1: Pfad-Ermittlung
    PROJECT_ANKER = "offline_ai"
    TARGET_ORDNER = "temp"
    
    temp_dir = find_project_path(anchor_name=PROJECT_ANKER, target_name=TARGET_ORDNER)
    cache_file_path = os.path.join(temp_dir, "automated_universe_cache.pkl")
    print(f"[Anschluss 1] Ziel-Verzeichnis gesetzt auf: {cache_file_path}")
    
    # KLEMME 2: Bestandsaufnahme
    cached_data = {}
    if os.path.exists(cache_file_path) and os.path.getsize(cache_file_path) > 0:
        try:
            with open(cache_file_path, 'rb') as f:
                cached_data = pickle.load(f)
            print(f"[Anschluss 2] Lokaler Altbestand geladen: {len(cached_data)} Assets im Speicher.")
        except Exception as e:
            print(f"[!] Fehler beim Lesen der Cache-Datei: {e}. Starte mit leerem Speicher.")
            cached_data = {}
    else:
        print("[Anschluss 2] Kein Altbestand gefunden oder Datei ist leer. Initialer Start.")
            
    # KLEMME 3: Zustandsprüfungen
    is_today = _is_cache_from_today(cache_file_path)
    is_online = _has_internet()
    print(f"[Anschluss 3] Status-Check: Datei von heute? {is_today} | Internet vorhanden? {is_online}")
    
    if not is_online:
        print("[Anschluss 5] System arbeitet OFFLINE. Signal bricht ab. Nutze reinen Altbestand.")
        return {t: data for t, data in cached_data.items() if data is not None}
        
    # KLEMME 4: Mindestmenge bestimmen über die Scout-Funktion
    print("[Anschluss 4] Scouten der Ticker-Begriffe...")
    active_tickers = _discover_active_markets(market_switches)
    today_str = datetime.now().strftime("%Y-%m-%d")
    
    # KLEMME 5: Die Paket-Weiche (Zentrale Download-Steuerung)
    print("[Anschluss 5] Analysiere Ticker-Zustände für Paket-Verarbeitung...")
    
    # Ticker filtern, die noch komplett im Cache fehlen (Erst-Download / Reparatur)
    neue_ticker = [t for t in active_tickers if t not in cached_data]
    
    # --- STRANG A: ERST-DOWNLOAD (Der schnelle Bulk-Container) ---
    if neue_ticker:
        print(f"   -> {len(neue_ticker)} neue Ticker erkannt. Starte Bulk-Erstladung...")
        bulk_prices = fetch_bulk_price_data(neue_ticker, period="5y")
        
        if bulk_prices is not None:
            for ticker in neue_ticker:
                # Logik-Korrektur: Holt das bereits formatierte DataFrame direkt aus dem Dictionary
                ticker_df = bulk_prices.get(ticker)
                
                if ticker_df is not None and not ticker_df.empty:
                    print(f"      -> Erstelle Fundamentaldaten-Payload für: {ticker}")
                    fundamentals = fetch_single_fundamentals(ticker, active_tickers[ticker])
                    
                    if fundamentals:
                        cached_data[ticker] = {
                            'prices': ticker_df,
                            **fundamentals  # Fusioniert Preise und Bilanzen in ein Dict
                        }
                    apply_rate_limit()  # Pause nur für den tiefen Fundamentals-Call nötig
                else:
                    # Ticker lieferte leeres DataFrame oder None -> Tot-Flag setzen
                    print(f"      [i] Ticker {ticker} ungültig oder leer bei Yahoo. Setze Tot-Flag (None).")
                    cached_data[ticker] = None
            
            # Physische Sicherung direkt nach dem Bulk-Vorgang auf die Festplatte brennen
            save_live_directory(cache_file_path, cached_data)

    # --- STRANG B: INKREMENTELLES PAKET-UPDATE (Delta für valide Bestände) ---
    valide_ticker = [t for t in active_tickers if t in cached_data and cached_data[t] is not None]
    
    if valide_ticker and not is_today:
        print(f"   -> Cache nicht von heute. Prüfe Delta-Bedarf für {len(valide_ticker)} Assets...")
        
        # Ermittle das älteste aufgezeichnete Datum im Cache, um die Lücke zu bestimmen
        min_last_date = None
        for t in valide_ticker:
            df = cached_data[t]['prices']
            if not df.empty:
                current_max = df.index.max().date()
                if min_last_date is None or current_max < min_last_date:
                    min_last_date = current_max
        
        # Wenn Lücke vorhanden, ziehen wir das Paket-Delta
        if min_last_date and min_last_date < datetime.now().date():
            start_delta = (min_last_date + datetime.timedelta(days=1)).strftime("%Y-%m-%d")
            print(f"   -> Lade Paket-Delta ab {start_delta} bis heute ({today_str}) für alle Assets...")
            
            delta_prices = fetch_bulk_price_data(valide_ticker, start_date=start_delta, end_date=today_str)
            
            if delta_prices is not None:
                for ticker in valide_ticker:
                    # Logik-Korrektur: Direkter Dictionary-Zugriff statt Multi-Index Abfrage
                    ticker_delta_df = delta_prices.get(ticker)
                    
                    if ticker_delta_df is not None and not ticker_delta_df.empty:
                        old_df = cached_data[ticker]['prices']
                        # Zusammenfügen und Duplikate eliminieren
                        cached_data[ticker]['prices'] = pd.concat([old_df, ticker_delta_df]).drop_duplicates()
                
                # Delta-Update physisch auf Disk verriegeln
                save_live_directory(cache_file_path, cached_data)
        else:
            print("   [✓] Lokale Kurshistorie ist bereits vollkommen deckungsgleich mit dem aktuellen Handelstag.")
    elif is_today:
        print("   [✓] Inkrementelles Update übersprungen. Cache-Datei wurde heute bereits aktualisiert.")
        
    # KLEMME 6: Signal-Ausgang (Filtert alle None-Leichen für die RAM-Bereitstellung heraus)
    universe_data = {t: data for t, data in cached_data.items() if data is not None}
    print(f"[ENDE] Pipeline erfolgreich verklemmt. {len(universe_data)} Assets stehen im RAM bereit.")
    
    return universe_data

# Der korrekte, sichere Trigger beim Drücken von Play:
mein_markt_universum = run_market_pipeline_download(market_switches)

[START] Starte globale Markt-Pipeline (Bulk-Architektur)
[✓] Projekthauptordner lokalisiert unter: /Users/cristallagus/Desktop/GitHub/Offline_AI
[✓] Existierenden Ordner 'temp' im System gefunden: /Users/cristallagus/Desktop/GitHub/Offline_AI/temp
[Anschluss 1] Ziel-Verzeichnis gesetzt auf: /Users/cristallagus/Desktop/GitHub/Offline_AI/temp/automated_universe_cache.pkl
[Anschluss 2] Kein Altbestand gefunden oder Datei ist leer. Initialer Start.
[Anschluss 3] Status-Check: Datei von heute? False | Internet vorhanden? True
[Anschluss 4] Scouten der Ticker-Begriffe...
   [Scout] AKTIEN: Schalter steht auf 0 (Unbegrenzt). 503 Begriffe bereitgestellt.
[Anschluss 5] Analysiere Ticker-Zustände für Paket-Verarbeitung...
   -> 503 neue Ticker erkannt. Starte Bulk-Erstladung...
   [API Bulk] Starte Paket-Download für 503 Ticker...
      -> Erstelle Fundamentaldaten-Payload für: MMM
   [i] Rate Limiting: Pausiere für 2.51 Sekunden...
      -> Erstelle Fundamentaldaten-Payload für: AOS
   [i] Rate

KeyboardInterrupt: 

In [ ]:
print(f"Aktueller Pfad: {os.getcwd()}")
print(f"Zielordner existiert: {os.path.exists('../../temp')}")

In [ ]:
class MarketValuationAnalyzer:
    def __init__(self, source_cache_dir: str = "../../temp"):
        self.db_file = get_knowledge_db_path("app_data_business_analytics.db")
        self._init_db()

    def _init_db(self) -> None:
        """Initialisiert die Datenbank tabellensicher."""
        with sqlite3.connect(self.db_file) as conn:
            cursor = conn.cursor()
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS asset_metadata (
                    ticker TEXT PRIMARY KEY, 
                    name TEXT, 
                    sector TEXT, 
                    industry TEXT, 
                    asset_category TEXT
                )
            """)
            # Datenbank wird um stochastic_k und stochastic_d erweitert
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS historical_analyses (
                    ticker TEXT, date_stamp TEXT,
                    current_price REAL, historical_lowest_price REAL, historical_lowest_date TEXT, pct_above_lowest REAL,
                    kgv_current REAL, ev_ebitda REAL, kuv_current REAL, kuv_at_historical_lowest REAL,
                    gross_margin_avg REAL, operating_margin_avg REAL, net_margin_avg REAL, roe_current REAL, roic_current REAL,
                    debt_to_equity REAL, current_ratio REAL, quick_ratio REAL,
                    sma_50 REAL, sma_200 REAL, ema_20 REAL,
                    rsi_14 REAL, macd_line REAL, macd_signal REAL, bb_upper REAL, bb_lower REAL,
                    stochastic_k REAL, stochastic_d REAL, -- NEU & DYNAMISCH
                    beta REAL, value_at_risk_95 REAL, sharpe_ratio REAL,
                    intrinsic_value REAL, risk_reward_ratio TEXT, investment_decision TEXT, target_price_12m REAL,
                    last_updated TEXT,
                    PRIMARY KEY (ticker, date_stamp)
                )
            """)
            conn.commit()

    def calculate_stochastic(self, prices_df: pd.DataFrame, k_period: int = 14, d_period: int = 3) -> tuple:
        """Berechnet Stochastik-Werte für den aktuellsten Zeitschritt."""
        try:
            if len(prices_df) < k_period:
                return None, None
            
            # Tiefstes Tief / Höchstes Hoch über das k-Perioden-Fenster
            low_min = prices_df['Low'].rolling(window=k_period).min()
            high_max = prices_df['High'].rolling(window=k_period).max()
            
            # %K berechnen
            denom = (high_max - low_min).replace(0, np.nan)
            k_series = ((prices_df['Close'] - low_min) / denom) * 100
            
            # %D berechnen (SMA von %K)
            d_series = k_series.rolling(window=d_period).mean()
            
            # Den jeweils aktuellsten (letzten) Wert zurückgeben
            return float(k_series.iloc[-1]), float(d_series.iloc[-1])
        except Exception:
            return None, None

    def run_analysis(self, universe_data: dict) -> None:
        with sqlite3.connect(self.db_file) as conn:
            cursor = conn.cursor()
            date_stamp = datetime.now().strftime('%Y-%m-%d')
            ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

            for ticker, data in universe_data.items():
                if ticker == "^GSPC" or not isinstance(data, dict): 
                    continue
                    
                info = data.get('info', {})
                prices = data.get('prices', pd.DataFrame())
                if prices.empty: 
                    continue

                # 1. Dynamische Berechnungen (Kein Fix-Coding)
                close = prices['Close'].dropna()
                curr, low = float(close.iloc[-1]), float(close.min())
                rsi14 = self._calc_rsi(close)
                sma50 = float(close.rolling(50).mean().iloc[-1]) if len(close) >= 50 else None
                
                # Dynamische Stochastik berechnen
                stoch_k, stoch_d = self.calculate_stochastic(prices)
                
                # 2. Kennzahlen auslesen
                gross_m = float(info.get('grossMargins', 0)) * 100 if info.get('grossMargins') else None
                oper_m = float(info.get('operatingMargins', 0)) * 100 if info.get('operatingMargins') else None
                net_m = float(info.get('profitMargins', 0)) * 100 if info.get('profitMargins') else None
                roe = float(info.get('returnOnEquity', 0)) * 100 if info.get('returnOnEquity') else None
                
                # Dynamische Handelsentscheidung: Verknüpfung von RSI und Stochastik
                decision = "HALTEN"
                if rsi14 and rsi14 < 30 and stoch_k and stoch_k < 20: 
                    decision = "KAUFEN"  # Überverkauft in beiden Indikatoren
                elif rsi14 and rsi14 > 70 and stoch_k and stoch_k > 80: 
                    decision = "VERKAUFEN"  # Überkauft in beiden Indikatoren

                # 3. In Datenbank sichern
                cursor.execute("INSERT OR IGNORE INTO asset_metadata (ticker, name) VALUES (?, ?)", 
                               (ticker, info.get('longName', 'Unknown')))

                # 34 Spalten Query befüllen (Inklusive Stochastik)
                cursor.execute("""
                    INSERT OR REPLACE INTO historical_analyses VALUES 
                    (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
                """, (
                    ticker, date_stamp, curr, low, close.idxmin().strftime('%Y-%m-%d'), ((curr/low)-1)*100,
                    info.get('trailingPE'), info.get('enterpriseToEbitda'), info.get('priceToSalesTrailing12Months'), None,
                    gross_m, oper_m, net_m, roe, None, 
                    info.get('debtToEquity'), info.get('currentRatio'), info.get('quickRatio'),
                    sma50, None, None, rsi14, None, None, None, None, 
                    stoch_k, stoch_d,  # NEUE WERTE HIER INTEGRIERT
                    None, None, None, None, "MEDIUM", decision, None, ts
                ))
            conn.commit()

## 3. Quantitative Fundamental Analysis
Goal: Evaluate the financial health and intrinsic value of the business.

In [ ]:
# Ordner und SQL ablage
def get_knowledge_db_path(db_name: str = "app_data_business_analytics.db", target_folder_name: str = "knowledge") -> str:
    """
    Wandert im Verzeichnisbaum ausgehend vom aktuellen Skript/Notebook nach oben,
    sucht den echten Hauptordner (z.B. 'knowledge') und gibt den absoluten Pfad 
    zur gewünschten SQL-Datenbank zurück. Verhindert das Duplizieren von Ordnern.
    """
    import os
    
    # Startpunkt bestimmen (Unterstützt sowohl .py-Skripte als auch Jupyter Notebooks)
    current_dir = os.path.abspath(os.path.dirname(__file__)) if '__file__' in globals() else os.path.abspath(os.getcwd())
    
    while True:
        potential_path = os.path.join(current_dir, target_folder_name)
        # Wenn der Ordner existiert, haben wir die Zentrale gefunden
        if os.path.isdir(potential_path):
            return os.path.join(potential_path, db_name)
        
        # Eine Ebene nach oben wandern
        parent_dir = os.path.dirname(current_dir)
        
        # Falls das Root-Verzeichnis erreicht wurde (Abbruchbedingung): Fallback
        if parent_dir == current_dir:
            fallback_dir = os.path.join(os.path.abspath(os.getcwd()), target_folder_name)
            os.makedirs(fallback_dir, exist_ok=True)
            return os.path.join(fallback_dir, db_name)
            
        current_dir = parent_dir

### 3.1 Valuation Metrics
P/E Ratio (Price-to-Earnings) vs. Historical Average & Peers

EV/EBITDA for capital-structure neutral valuation

P/S Ratio (Price-to-Sales) for growth-stage evaluation

### 3.2 Profitability & Capital Efficiency
Gross / Operating / Net Margins (Trend analysis over 5 years)

ROE (Return on Equity) & ROIC (Return on Invested Capital)

### 3.3 Financial Leverage & Solvency
Debt-to-Equity Ratio

Current / Quick Ratio (Short-term liquidity)

## 4. Technical Analysis & Momentum Indicators
Goal: Identify optimal entry/exit points and market sentiment.

### 4.1 Trend & Moving Averages
-   SMA 50 vs. SMA 200 (Golden Cross / Death Cross tracking)

-   Exponential Moving Averages (EMA) for short-term momentum

### 4.2 Volatility & Momentum Oscillators
-   RSI (Relative Strength Index): Identifying overbought (< 70) or oversold (> 30) conditions

-   MACD (Moving Average Convergence Divergence): Trend reversal tracking

-   Bollinger Bands: Price volatility bands

## 5. Statistical Risk & Portfolio Integration
Goal:Measure risk metrics relative to the broader market.
-   Beta ($\beta$): Systematic risk calculation relative to S&P 500 / DAX
-   Value at Risk (VaR): Maximum expected loss under normal market conditions
-   Sharpe Ratio: Risk-adjusted return performance

## 6. Final Evaluation & Actionable Insights
Goal: Synthesize fundamental and technical data into a definitive rating.

-   Intrinsic Value Estimate: (e.g., via a simple Discounted Cash Flow or Multiples approach)

-   Risk/Reward Rating: Low / Medium / High

-   Investment Decision: BUY / HOLD / SELL

-   Target Price: [Target Price 12-Month Horizon]

In [ ]:
# Calculation
def get_knowledge_db_path(db_name: str) -> str:
    return os.path.join("../../knowledge", db_name)

class MarketValuationAnalyzer:
    def __init__(self, source_cache_dir: str = "../../temp"):
        self.db_file = get_knowledge_db_path("app_data_business_analytics.db")
        self._init_db()

    def _init_db(self) -> None:
        with sqlite3.connect(self.db_file) as conn:
            cursor = conn.cursor()
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS asset_metadata (
                    ticker TEXT PRIMARY KEY, 
                    name TEXT, 
                    sector TEXT, 
                    industry TEXT, 
                    asset_category TEXT
                )
            """)
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS historical_analyses (
                    ticker TEXT, date_stamp TEXT,
                    current_price REAL, historical_lowest_price REAL, historical_lowest_date TEXT, pct_above_lowest REAL,
                    kgv_current REAL, ev_ebitda REAL, kuv_current REAL, kuv_at_historical_lowest REAL,
                    gross_margin_avg REAL, operating_margin_avg REAL, net_margin_avg REAL, roe_current REAL, roic_current REAL,
                    debt_to_equity REAL, current_ratio REAL, quick_ratio REAL,
                    sma_50 REAL, sma_200 REAL, ema_20 REAL,
                    rsi_14 REAL, macd_line REAL, macd_signal REAL, bb_upper REAL, bb_lower REAL,
                    beta REAL, value_at_risk_95 REAL, sharpe_ratio REAL,
                    intrinsic_value REAL, risk_reward_ratio TEXT, investment_decision TEXT, target_price_12m REAL,
                    last_updated TEXT,
                    PRIMARY KEY (ticker, date_stamp)
                )
            """)
            conn.commit()

    def _calc_rsi(self, s, p=14):
        delta = s.diff()
        gain = delta.where(delta > 0, 0).rolling(p).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(p).mean()
        rs = gain / loss.replace(0, np.nan)
        return float(100 - (100 / (1 + rs.iloc[-1]))) if not rs.empty else None

    def run_analysis(self, universe_data: dict) -> None:
        with sqlite3.connect(self.db_file) as conn:
            cursor = conn.cursor()
            date_stamp = datetime.now().strftime('%Y-%m-%d')
            ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

            for ticker, data in universe_data.items():
                if ticker == "^GSPC" or not isinstance(data, dict): continue
                info = data.get('info', {})
                prices = data.get('prices', pd.DataFrame())
                if prices.empty: continue

                # 1. Berechnungen
                close = prices['Close'].dropna()
                curr, low = float(close.iloc[-1]), float(close.min())
                rsi14 = self._calc_rsi(close)
                sma50 = float(close.rolling(50).mean().iloc[-1]) if len(close) >= 50 else None
                
                # 2. Kennzahlen (ohne Datenverlust)
                gross_m = float(info.get('grossMargins', 0)) * 100 if info.get('grossMargins') else None
                oper_m = float(info.get('operatingMargins', 0)) * 100 if info.get('operatingMargins') else None
                net_m = float(info.get('profitMargins', 0)) * 100 if info.get('profitMargins') else None
                roe = float(info.get('returnOnEquity', 0)) * 100 if info.get('returnOnEquity') else None
                
                decision = "HALTEN"
                if rsi14 and rsi14 < 35 and roe and roe > 15: decision = "KAUFEN"
                elif rsi14 and rsi14 > 75: decision = "VERKAUFEN"

                # 3. Metadaten (Sicherung)
                cursor.execute("INSERT OR IGNORE INTO asset_metadata (ticker, name) VALUES (?, ?)", 
                               (ticker, info.get('longName', 'Unknown')))

                # 4. Finales Insert (34 Spalten)
                cursor.execute("""
                    INSERT OR REPLACE INTO historical_analyses VALUES 
                    (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
                """, (
                    ticker, date_stamp, curr, low, close.idxmin().strftime('%Y-%m-%d'), ((curr/low)-1)*100,
                    info.get('trailingPE'), info.get('enterpriseToEbitda'), info.get('priceToSalesTrailing12Months'), None,
                    gross_m, oper_m, net_m, roe, None, 
                    info.get('debtToEquity'), info.get('currentRatio'), info.get('quickRatio'),
                    sma50, None, None, rsi14, None, None, None, None, 
                    None, None, None, None, "MEDIUM", decision, None, ts
                ))
            conn.commit()

# AUSFÜHRUNG (Genau wie besprochen)
with open("../../temp/automated_universe_cache.pkl", 'rb') as f:
    universe_data = pickle.load(f)

analyzer = MarketValuationAnalyzer()
analyzer.run_analysis(universe_data)

In [ ]:
# Test
df_file_db = get_knowledge_db_path("app_data_business_analytics.db")

conn = sqlite3.connect(df_file_db)

# KORREKTUR: Tabelle heißt jetzt 'historical_analyses'
query = """
    SELECT 
        h.ticker, 
        m.name, 
        h.kgv_current, 
        h.gross_margin_avg, 
        h.roe_current, 
        h.debt_to_equity,
        h.investment_decision
    FROM historical_analyses h
    LEFT JOIN asset_metadata m ON h.ticker = m.ticker
    WHERE h.date_stamp = (SELECT MAX(date_stamp) FROM historical_analyses)
    LIMIT 10
"""

df_test = pd.read_sql_query(query, conn)
conn.close()

# Ergebnis anzeigen
print(df_test)

In [ ]:
# Test2
cache_datei_pfad = "../../temp/automated_universe_cache.pkl"

with open(cache_datei_pfad, 'rb') as f:
    meine_ram_daten = pickle.load(f)

# Wir picken uns eine Aktie heraus, die im Cache existiert
test_ticker = "AAPL" 

if test_ticker in meine_ram_daten:
    info_keys = meine_ram_daten[test_ticker].get('info', {}).keys()
    print(f"--- Verfügbare Keys in 'info' für {test_ticker} ---")
    
    # Filtere nach Begriffen wie Margin, Debt, Equity, Return
    interessante_keys = [k for k in info_keys if any(x in k.lower() for x in ['margin', 'debt', 'equity', 'return', 'ratio'])]
    for key in sorted(interessante_keys):
        value = meine_ram_daten[test_ticker]['info'][key]
        print(f"{key}: {value}")
else:
    print(f"Ticker {test_ticker} nicht im Cache gefunden.")